In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')


Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


Bolsa de Palabras

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
 #Parámetros predeterminados para instanciar el objeto de segmentación de palabras
vec=CountVectorizer()


In [ ]:
## Uso para tokenizar el corpus de texto y contar la frecuencia de palabras:
corpus = [
  'This is the first document.',
  'This is the second second document.',
  'And the third one.',
    'Is this the first document?',
 ]

In [ ]:
X = vec.fit_transform(corpus)
 # A cada elemento encontrado durante el ajuste se le asigna un índice entero único correspondiente a la columna en la matriz resultante
X

<4x9 sparse matrix of type '<class 'numpy.int64'>'
	with 19 stored elements in Compressed Sparse Row format>

In [ ]:
 # Obtener nombre de la función (nombre de columna)
vec.get_feature_names()



/usr/local/lib/python3.7/dist-packages/sklearn/utils/deprecation.py:87: FutureWarning: Function get_feature_names is deprecated; get_feature_names is deprecated in 1.0 and will be removed in 1.2. Please use get_feature_names_out instead.
  warnings.warn(msg, category=FutureWarning)


['and', 'document', 'first', 'is', 'one', 'second', 'the', 'third', 'this']

In [ ]:
#Imprimo el array completo
X.toarray()


array([[0, 1, 1, 1, 0, 0, 1, 0, 1],
       [0, 1, 0, 1, 0, 2, 1, 0, 1],
       [1, 0, 0, 0, 1, 0, 1, 1, 0],
       [0, 1, 1, 1, 0, 0, 1, 0, 1]])

In [ ]:
 #Ver los parámetros del modelo de bolsa de palabras
vec.get_params()

In [ ]:
vec=CountVectorizer(ngram_range=(1, 2))
X = vec.fit_transform(corpus)
vec.get_feature_names()




TfIdf

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# lista de documentos de texto
text = ["The quick brown fox jumped over the lazy dog.",
 "The dog.",
 "The fox"]
# crear la transformación
vectorizer = TfidfVectorizer()
# tokenizar y construir vocabulario
vectorizer.fit(text)
# resumir
print(vectorizer.vocabulary_)
print(vectorizer.idf_)

{'the': 7, 'quick': 6, 'brown': 0, 'fox': 2, 'jumped': 3, 'over': 5, 'lazy': 4, 'dog': 1}
[1.69314718 1.28768207 1.28768207 1.69314718 1.69314718 1.69314718
 1.69314718 1.        ]


In [ ]:
# documento codificado
vector = vectorizer.transform(text)
# resumir vector codificado

print(vector.toarray())

[[0.36388646 0.27674503 0.27674503 0.36388646 0.36388646 0.36388646
  0.36388646 0.42983441]
 [0.         0.78980693 0.         0.         0.         0.
  0.         0.61335554]
 [0.         0.         0.78980693 0.         0.         0.
  0.         0.61335554]]


# Clasificador Naive Bayes

Vamos a aplicar un modelo para clasificar el sentimiento de los comentarios de usuarios en las redes sociales de una marca

In [ ]:
import numpy as np, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import  train_test_split
import glob


In [ ]:
files= glob.glob(r'/content/gdrive/MyDrive/Curso de Minería de Textos/Datasets/Menciones/*.xls*')

df= pd.DataFrame()
for i in range(0,len(files)):
            df_1 = pd.read_excel(files[i])
            df = pd.concat([df_1,df])
df.head()

,Message,Sentimiento
0,"@DIRECTVGOAyuda Hola, tengo una serie de pregu...",Neutro
1,@DIRECTVGO Hola @DIRECTVGO hice la compra del ...,Negativo
2,"@DIRECTVGOAyuda Hola, ¿por qué me sale ""Fallo ...",Negativo
3,"@DIRECTVGOAyuda Ayer entré normalmente, pude a...",Negativo
4,es que ayer podia entrar perfectamente a Direc...,Negativo


In [ ]:
df_test= df[df.Sentimiento.isna()]
df= df[~df.Sentimiento.isna()]

In [ ]:
df_test

,Message,Sentimiento
0,Cómo hago para cancelar esa inscripción,NaN
1,El CDF premium no se ve en DirecTVgo.,NaN
2,En directv go está disponible el canal cdf per...,NaN
3,@DIRECTVChile una consulta por qué en la aplic...,NaN
4,"@DIRECTVGO De Chile, 2 dias sin poder utilizar...",NaN
...,...,...
3268,@DIRECTVGO Me gusta tener DIRECTV GO,NaN
3269,dios bendiga directvgo,NaN
3270,a mi me funciona de maravilla ♥️♥️♥️,NaN
3271,@DIRECTVServicio Hola ¿cómo se puede agregar s...,NaN


# Opción 1: Preprocesamiento con NLTK


Esta función construye un pípeline con todos los pasos que vimos en la Unidad de Preprocesamiento

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
import re
import string
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
stopWords = set(stopwords.words('spanish'))
from nltk.stem import PorterStemmer
default_stemmer = PorterStemmer()

default_stopwords = stopwords.words('spanish')
def clean_text(text, ):

    def tokenize_text(text):
        return [w for s in sent_tokenize(text) for w in word_tokenize(s)]

    def remove_special_characters(text, characters=string.punctuation.replace('-', '')):
        tokens = tokenize_text(text)
        pattern = re.compile('[{}]'.format(re.escape(characters)))
        return ' '.join(filter(None, [pattern.sub('', t) for t in tokens]))

    def stem_text(text, stemmer=default_stemmer):
        tokens = tokenize_text(text)
        return ' '.join([stemmer.stem(t) for t in tokens])

    def remove_stopwords(text, stop_words=default_stopwords):
        tokens = [w for w in tokenize_text(text) if w not in stop_words]
        return ' '.join(tokens)

    text = text.strip(' ')
    text = text.lower()
    text = stem_text(text)
    text = remove_special_characters(text)
    text = remove_stopwords(text)

    return text



[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [ ]:
#Corremos la función

message_limpio = []
for i in df['Message']:
  message_limpio.append(clean_text(str(i)))

# Modelo

In [ ]:
## Label-Encoding  (Codificación de etiquetas manual)

serie_pos = df.Sentimiento.dropna().apply(lambda x: 1 if 'osit' in x else 0)
# 1️⃣ Elimina los valores NaN de la columna "Sentimiento".
# 2️⃣ Recorre cada valor (x) con apply() y:
#     - Si contiene la subcadena "osit" (como en "positivo"), devuelve 1.
#     - Si no, devuelve 0.
# Resultado: una serie con 1 donde el texto tiene "positivo", 0 en caso contrario.

serie_neut = df.Sentimiento.dropna().apply(lambda x: 2 if 'eut' in x else 0)
# Aplica la misma lógica pero buscando "eut" (de "neutro").
# Asigna el valor 2 a los sentimientos neutros.

serie_neg = df.Sentimiento.dropna().apply(lambda x: 3 if 'eg' in x else 0)
# De igual forma, busca la subcadena "eg" (de "negativo") y asigna 3 si la encuentra.

serie_final = serie_pos + serie_neut + serie_neg
# Suma las tres series posición a posición.
# Como solo una de las tres condiciones será verdadera por fila, el resultado final será:
#   1 → positivo
#   2 → neutro
#   3 → negativo
#   0 → sin coincidencia (por si alguna cadena no cumple ninguna condición)


In [ ]:
df['sentimiento'] = serie_final
df['Message'] = [str (item) for item in message_limpio]
df = df[df['sentimiento'] != 0]

df = df.dropna()

In [ ]:
df.groupby('sentimiento').count()

,Message,Sentimiento
sentimiento,,
1,538,538
2,1076,1076
3,645,645


In [ ]:
postivos.shape[0]

538

In [ ]:
postivos = df[df['sentimiento'] == 1]
neutros = df[df['sentimiento'] == 2]
negativos = df[df['sentimiento'] == 3]

#Opción >>> Realizar subsampling para equilibrar la muestra
df = pd.concat([postivos,negativos, neutros.sample(postivos.shape[0]*2)]) #El tamaño del sample es el tamaño de datos disponibles de Positivo.

Dividimos en Entrenamiento y Test

In [ ]:
#Train Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['Message'], df['sentimiento'], test_size = 0.20, stratify=df['sentimiento'], random_state = 12)

In [ ]:
#Entreno el modelo
model = make_pipeline(TfidfVectorizer(), MultinomialNB())

model.fit(X_train, y_train)


y_pred = model.predict(X_test)

In [ ]:
#Plot de la matriz de confusión
print(accuracy_score(y_test, y_pred))

0.8429203539823009


In [ ]:
from sklearn.metrics import confusion_matrix as cm
import pandas as pd

confusion_matrix=cm(y_test, y_pred)

list1 = ["Actual 1", "Actual 2", "Actual 3"]
list2 = ["Predicted 1", "Predicted 2", "Predicted 3"]
pd.DataFrame(confusion_matrix, list1,list2)


,Predicted 1,Predicted 2,Predicted 3
Actual 1,103,5,0
Actual 2,14,170,31
Actual 3,1,20,108


Probamos el modelo con el dataset no clasificado

In [ ]:
#Aplicamos mismo preprocesamiento que al conjunto de Train!!!
message_limpio_test = []
for i in df_test['Message']:
  message_limpio_test.append(clean_text(str(i)))


df_test['Message_clean'] = [str (item) for item in message_limpio_test]


In [ ]:
#Sumo los valores predichos por el modelo en la columna Sentiment

y_pred_sen = model.predict(df_test['Message_clean'])
df_test['Sentiment'] = y_pred_sen

Chequeamos si las predcciones hacen sentido

In [ ]:
df_test.loc[:,['Message', 'Sentiment']][df_test['Sentiment'] == 3][140:141]

,Message,Sentiment
786,"al llamar, no me deja, me considera como pre p...",3


In [ ]:
#Cómo se distribuyó la predicción

df_test.groupby('Sentiment').count()['Message']

Sentiment
1     168
2    2386
3     582
Name: Message, dtype: int64

# Clasificación con Random Forest aplicando Grid Search

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.svm import LinearSVC
from tempfile import mkdtemp
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from glob import glob
from sklearn import preprocessing
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import ExtraTreesClassifier
from joblib import dump, load

cache = mkdtemp()

pipe = Pipeline([
    ('vec', CountVectorizer()),
    ('clf', RandomForestClassifier())], memory=cache)


params = [{'clf': [MultinomialNB()],
           },

          {'clf': [RandomForestClassifier()], 'clf__n_estimators':[100, 200, 500],
           'clf__max_depth': [5, 10, 15],
           'clf__min_samples_split': [5, 10, 15],
           'vec': [CountVectorizer(),
            TfidfVectorizer()]
           }
         ]

gs = GridSearchCV(estimator=pipe, param_grid=params, cv=5, n_jobs=-1, verbose=10)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['Message'], df['sentimiento'], test_size = 0.20, stratify=df['sentimiento'], random_state = 12)


gs.fit(X_train, y_train)
y_pred = gs.predict(X_test)


#joblib es un módulo de python que nos permite serializar objetos, es decir, guardar en un archivo cualquier objeto que hayamos instanciado. Tiene dos métodos básicos: dump() y load()

dump(gs, 'modelo_dtv.joblib')

print(classification_report(y_test, y_pred))

Fitting 5 folds for each of 55 candidates, totalling 275 fits
              precision    recall  f1-score   support

           1       0.88      0.95      0.92       108
           2       0.91      0.68      0.78       215
           3       0.67      0.90      0.77       129

    accuracy                           0.81       452
   macro avg       0.82      0.84      0.82       452
weighted avg       0.83      0.81      0.81       452



In [ ]:
gs.best_estimator_

In [ ]:
gs2 = load('modelo_dtv.joblib')

y_pred_sen = gs2.predict(df_test['Message'])

df_test['Sentiment'] = y_pred_sen

In [ ]:
df_test.groupby('Sentiment').count()